# 🔍 Проверка баланса на всех этапах

**Цель**: Проверить, что баланс (Assets = Liabilities + Equity) сходится на всех этапах:
1. Официальная отчетность
2. RAW БД (после загрузки из Edgar CSV)
3. CANONICAL БД (после normalize_to_canonical)
4. **Иерархическая каноническая структура (CanonicalFinancialStatements)** ⭐
5. Препроцессинг (HistoricState)
6. Движок модели (ThreeStatementModel)

**Формула проверки**: `Total Assets = Total Liabilities + Total Equity`


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("../../..").resolve()
COMPANY = "us_steel"
croot = ROOT / "companies" / COMPANY

sys.path.insert(0, str(ROOT))

from engine.database.data_mart import get_data_mart
from engine.model.preprocess import ModelPreprocessor
from engine.model3.core import ThreeStatementModel
from engine.model3.io import load_inputs

# Импорт для иерархической канонической структуры
try:
    from engine.model3.canonical import CanonicalFinancialStatements
    CANONICAL_AVAILABLE = True
except ImportError:
    CANONICAL_AVAILABLE = False
    print("⚠️ Иерархическая каноническая структура недоступна (canonical.py не найден)")

print(f"✅ Импорты выполнены")
print(f"   ROOT: {ROOT}")
print(f"   COMPANY: {COMPANY}")
print(f"   CanonicalFinancialStatements доступен: {CANONICAL_AVAILABLE}")


## 📊 ШАГ 1: Официальная отчетность (2024)


In [ ]:
# Официальные данные BS 2024 из ноутбука 01_Fix_Canonical_Format_BS.ipynb
official_bs_2024 = {
    'total_assets': 18_607_000_000,
    'total_current_assets': 4_799_000_000,
    'total_non_current_assets': 13_808_000_000,
    'total_liabilities': 8_315_000_000,
    'total_current_liabilities': 3_188_000_000,
    'total_non_current_liabilities': 5_127_000_000,
    'total_equity': 10_292_000_000,
    'total_liab_equity': 18_607_000_000,
}

print("="*80)
print("📊 ШАГ 1: ОФИЦИАЛЬНАЯ ОТЧЕТНОСТЬ (2024)")
print("="*80)

assets_official = official_bs_2024['total_assets']
liab_official = official_bs_2024['total_liabilities']
equity_official = official_bs_2024['total_equity']
liab_equity_official = liab_official + equity_official
diff_official = assets_official - liab_equity_official

print(f"\n📋 Баланс из официальной отчетности:")
print(f"   Total Assets: ${assets_official:,.0f}")
print(f"   Total Liabilities: ${liab_official:,.0f}")
print(f"   Total Equity: ${equity_official:,.0f}")
print(f"   Liabilities + Equity: ${liab_equity_official:,.0f}")
print(f"   Diff (Assets - Liab+Equity): ${diff_official:,.0f}")

if abs(diff_official) < 1e-6:
    print(f"\n✅ Баланс сходится!")
else:
    print(f"\n❌ Баланс НЕ сходится! Разница: ${diff_official:,.0f}")


## 📊 ШАГ 2: RAW БД (после загрузки из Edgar CSV)


In [ ]:
mart = get_data_mart(ROOT, COMPANY)

# Загружаем RAW данные
bs_raw = mart.get_history_balance_sheet(canonical=False)

print("="*80)
print("📊 ШАГ 2: RAW БД (после загрузки из Edgar CSV)")
print("="*80)

# Получаем значения для 2024 года
year = 2024
year_str = str(year)

def get_raw_value(metric_name):
    """Получает значение метрики из RAW БД"""
    if bs_raw.empty:
        return None
    metric_row = bs_raw[bs_raw['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

# Суммируем активы
assets_raw = 0
assets_components = []
for metric in ['cash', 'accounts_receivable', 'inventory', 'other_current_assets', 
               'ppe_net', 'goodwill', 'intangible_assets_net', 'rou_asset', 
               'restricted_cash', 'investments_and_long_term_receivables', 
               'other_non_current_assets']:
    val = get_raw_value(metric)
    if pd.notna(val) and val != 0:
        assets_raw += val
        assets_components.append((metric, val))

# Проверяем total_assets
total_assets_raw = get_raw_value('total_assets')
if pd.notna(total_assets_raw):
    assets_raw = total_assets_raw

# Суммируем обязательства
liab_raw = 0
liab_components = []
for metric in ['accounts_payable', 'accounts_payable_related_parties', 'st_debt', 
               'accrued_taxes', 'lease_liab_current', 'other_current_liabilities',
               'long_term_debt', 'lease_liab_noncurrent', 'other_non_current_liabilities']:
    val = get_raw_value(metric)
    if pd.notna(val) and val != 0:
        liab_raw += val
        liab_components.append((metric, val))

# Проверяем total_liabilities
total_liab_raw = get_raw_value('total_liabilities')
if pd.notna(total_liab_raw):
    liab_raw = total_liab_raw

# Суммируем капитал
equity_raw = 0
equity_components = []
for metric in ['common_stock_par', 'apic', 'retained_earnings', 'treasury_stock', 'nci']:
    val = get_raw_value(metric)
    if pd.notna(val) and val != 0:
        equity_raw += val
        equity_components.append((metric, val))

# Проверяем total_equity
total_equity_raw = get_raw_value('total_equity')
if pd.notna(total_equity_raw):
    equity_raw = total_equity_raw

liab_equity_raw = liab_raw + equity_raw
diff_raw = assets_raw - liab_equity_raw

print(f"\n📋 Баланс из RAW БД:")
print(f"   Total Assets: ${assets_raw:,.0f}")
print(f"   Total Liabilities: ${liab_raw:,.0f}")
print(f"   Total Equity: ${equity_raw:,.0f}")
print(f"   Liabilities + Equity: ${liab_equity_raw:,.0f}")
print(f"   Diff (Assets - Liab+Equity): ${diff_raw:,.0f}")

if abs(diff_raw) < 1e-6:
    print(f"\n✅ Баланс сходится!")
else:
    print(f"\n❌ Баланс НЕ сходится! Разница: ${diff_raw:,.0f}")

# Сравнение с официальной отчетностью
diff_vs_official = assets_raw - assets_official
print(f"\n📊 Сравнение с официальной отчетностью:")
print(f"   Assets: ${assets_raw:,.0f} vs ${assets_official:,.0f} (diff: ${diff_vs_official:,.0f})")


## 📊 ШАГ 3: CANONICAL БД (после normalize_to_canonical)


In [ ]:
# Загружаем CANONICAL данные
bs_canonical = mart.get_history_balance_sheet(canonical=True)

print("="*80)
print("📊 ШАГ 3: CANONICAL БД (после normalize_to_canonical)")
print("="*80)

def get_canonical_value(metric_name):
    """Получает значение метрики из CANONICAL БД"""
    if bs_canonical.empty:
        return None
    metric_row = bs_canonical[bs_canonical['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

# Получаем значения
assets_canonical = get_canonical_value('total_assets')
liab_canonical = get_canonical_value('total_liabilities')
equity_canonical = get_canonical_value('total_equity')

if pd.isna(assets_canonical) or pd.isna(liab_canonical) or pd.isna(equity_canonical):
    print("⚠️ Не все метрики найдены в CANONICAL БД")
    print(f"   total_assets: {assets_canonical}")
    print(f"   total_liabilities: {liab_canonical}")
    print(f"   total_equity: {equity_canonical}")
else:
    liab_equity_canonical = liab_canonical + equity_canonical
    diff_canonical = assets_canonical - liab_equity_canonical

    print(f"\n📋 Баланс из CANONICAL БД:")
    print(f"   Total Assets: ${assets_canonical:,.0f}")
    print(f"   Total Liabilities: ${liab_canonical:,.0f}")
    print(f"   Total Equity: ${equity_canonical:,.0f}")
    print(f"   Liabilities + Equity: ${liab_equity_canonical:,.0f}")
    print(f"   Diff (Assets - Liab+Equity): ${diff_canonical:,.0f}")

    if abs(diff_canonical) < 1e-6:
        print(f"\n✅ Баланс сходится!")
    else:
        print(f"\n❌ Баланс НЕ сходится! Разница: ${diff_canonical:,.0f}")

    # Сравнение с RAW БД
    print(f"\n📊 Сравнение с RAW БД:")
    print(f"   Assets: ${assets_canonical:,.0f} vs ${assets_raw:,.0f} (diff: ${assets_canonical - assets_raw:,.0f})")
    print(f"   Liabilities: ${liab_canonical:,.0f} vs ${liab_raw:,.0f} (diff: ${liab_canonical - liab_raw:,.0f})")
    print(f"   Equity: ${equity_canonical:,.0f} vs ${equity_raw:,.0f} (diff: ${equity_canonical - equity_raw:,.0f})")


## 📊 ШАГ 3.5: Иерархическая каноническая структура (CanonicalFinancialStatements) ⭐


In [ ]:
# Загружаем иерархическую каноническую структуру
print("="*80)
print("📊 ШАГ 3.5: ИЕРАРХИЧЕСКАЯ КАНОНИЧЕСКАЯ СТРУКТУРА (CanonicalFinancialStatements)")
print("="*80)

canonical_hierarchical = mart.get_canonical_financial_statements(year=year)

if canonical_hierarchical is None:
    print("⚠️ Иерархическая каноническая структура недоступна (адаптеры не найдены или данные отсутствуют)")
    assets_hierarchical = None
    liab_hierarchical = None
    equity_hierarchical = None
else:
    # Получаем значения из иерархической структуры
    bs_hierarchical = canonical_hierarchical.balance_sheet
    
    # Активы
    assets_hierarchical = bs_hierarchical.assets.total_assets
    if assets_hierarchical is None:
        # Суммируем вручную если total_assets не заполнен
        current_assets = bs_hierarchical.assets.current
        noncurrent_assets = bs_hierarchical.assets.non_current
        assets_hierarchical = (
            (current_assets.cash_and_equivalents or 0) +
            (current_assets.restricted_cash or 0) +
            (current_assets.accounts_receivable_net or 0) +
            (current_assets.inventory_total or 0) +
            (current_assets.prepaid_expenses or 0) +
            (current_assets.other_current_assets or 0) +
            (noncurrent_assets.ppe_net or 0) +
            (noncurrent_assets.rou_finance_asset or 0) +
            (noncurrent_assets.rou_operating_asset or 0) +
            (noncurrent_assets.intangibles_finite_life or 0) +
            (noncurrent_assets.intangibles_indefinite_life or 0) +
            (noncurrent_assets.goodwill or 0) +
            (noncurrent_assets.longterm_investments or 0) +
            (noncurrent_assets.equity_method_investments or 0) +
            (noncurrent_assets.deferred_tax_assets or 0) +
            (noncurrent_assets.other_noncurrent_assets or 0)
        )
    
    # Обязательства
    liab_hierarchical = bs_hierarchical.liabilities.total_liabilities
    if liab_hierarchical is None:
        # Суммируем вручную если total_liabilities не заполнен
        current_liab = bs_hierarchical.liabilities.current
        noncurrent_liab = bs_hierarchical.liabilities.non_current
        liab_hierarchical = (
            (current_liab.st_debt or 0) +
            (current_liab.current_portion_of_lt_debt or 0) +
            (current_liab.accounts_payable or 0) +
            (current_liab.accrued_compensation or 0) +
            (current_liab.taxes_payable or 0) +
            (current_liab.interest_payable or 0) +
            (current_liab.current_finance_lease_liability or 0) +
            (current_liab.current_operating_lease_liability or 0) +
            (current_liab.other_current_liabilities or 0) +
            (noncurrent_liab.lt_debt_bonds or 0) +
            (noncurrent_liab.lt_debt_bank or 0) +
            (noncurrent_liab.lt_debt_other or 0) +
            (noncurrent_liab.noncurrent_finance_lease_liability or 0) +
            (noncurrent_liab.noncurrent_operating_lease_liability or 0) +
            (noncurrent_liab.deferred_tax_liabilities or 0) +
            (noncurrent_liab.other_noncurrent_liabilities or 0)
        )
    
    # Капитал
    equity_hierarchical = bs_hierarchical.equity.total_equity
    if equity_hierarchical is None:
        # Суммируем вручную если total_equity не заполнен
        equity_hierarchical = (
            (bs_hierarchical.equity.share_capital or 0) +
            (bs_hierarchical.equity.additional_paid_in_capital or 0) +
            (bs_hierarchical.equity.treasury_stock or 0) +
            (bs_hierarchical.equity.retained_earnings or 0) +
            (bs_hierarchical.equity.aoci or 0) +
            (bs_hierarchical.equity.noncontrolling_interests or 0)
        )
    
    liab_equity_hierarchical = liab_hierarchical + equity_hierarchical
    diff_hierarchical = assets_hierarchical - liab_equity_hierarchical
    
    print(f"\n📋 Баланс из иерархической канонической структуры:")
    print(f"   Total Assets: ${assets_hierarchical:,.0f}")
    print(f"   Total Liabilities: ${liab_hierarchical:,.0f}")
    print(f"   Total Equity: ${equity_hierarchical:,.0f}")
    print(f"   Liabilities + Equity: ${liab_equity_hierarchical:,.0f}")
    print(f"   Diff (Assets - Liab+Equity): ${diff_hierarchical:,.0f}")
    
    if abs(diff_hierarchical) < 1e-6:
        print(f"\n✅ Баланс сходится!")
    else:
        print(f"\n❌ Баланс НЕ сходится! Разница: ${diff_hierarchical:,.0f}")
    
    # Сравнение с CANONICAL БД (плоский формат)
    if pd.notna(assets_canonical):
        print(f"\n📊 Сравнение с CANONICAL БД (плоский формат):")
        print(f"   Assets: ${assets_hierarchical:,.0f} vs ${assets_canonical:,.0f} (diff: ${assets_hierarchical - assets_canonical:,.0f})")
        print(f"   Liabilities: ${liab_hierarchical:,.0f} vs ${liab_canonical:,.0f} (diff: ${liab_hierarchical - liab_canonical:,.0f})")
        print(f"   Equity: ${equity_hierarchical:,.0f} vs ${equity_canonical:,.0f} (diff: ${equity_hierarchical - equity_canonical:,.0f})")
    
    # Проверка детализации лизинга (если доступна)
    if bs_hierarchical.assets.non_current.rou_finance_asset is not None or bs_hierarchical.assets.non_current.rou_operating_asset is not None:
        rou_finance = bs_hierarchical.assets.non_current.rou_finance_asset or 0
        rou_operating = bs_hierarchical.assets.non_current.rou_operating_asset or 0
        rou_total = rou_finance + rou_operating
        print(f"\n💡 Детализация лизинга (ROU Asset):")
        print(f"   Finance Lease ROU Asset: ${rou_finance:,.0f}")
        print(f"   Operating Lease ROU Asset: ${rou_operating:,.0f}")
        print(f"   Total ROU Asset: ${rou_total:,.0f}")
    
    if (bs_hierarchical.liabilities.current.current_finance_lease_liability is not None or 
        bs_hierarchical.liabilities.current.current_operating_lease_liability is not None):
        lease_cur_finance = bs_hierarchical.liabilities.current.current_finance_lease_liability or 0
        lease_cur_operating = bs_hierarchical.liabilities.current.current_operating_lease_liability or 0
        lease_cur_total = lease_cur_finance + lease_cur_operating
        print(f"\n💡 Детализация лизинга (Current Lease Liability):")
        print(f"   Current Finance Lease Liability: ${lease_cur_finance:,.0f}")
        print(f"   Current Operating Lease Liability: ${lease_cur_operating:,.0f}")
        print(f"   Total Current Lease Liability: ${lease_cur_total:,.0f}")


## 📊 ШАГ 4: Препроцессинг (HistoricState)


In [ ]:
# Загружаем HistoricState через препроцессинг
preprocessor = ModelPreprocessor(ROOT, COMPANY)
historic_state = preprocessor.get_historic_state()

print("="*80)
print("📊 ШАГ 4: ПРЕПРОЦЕССИНГ (HistoricState)")
print("="*80)

# Суммируем активы из HistoricState
assets_preprocess = (
    historic_state.cash +
    historic_state.ar +
    historic_state.inventory +
    historic_state.other_current_assets +
    historic_state.ppe +
    historic_state.goodwill +
    historic_state.intangibles +
    historic_state.rou_asset +
    historic_state.other_non_current_assets
)

# Суммируем обязательства
liab_preprocess = (
    historic_state.st_debt +
    historic_state.ap +
    historic_state.other_current_liabilities +
    historic_state.debt +
    historic_state.lease_liability +
    historic_state.other_non_current_liabilities
)

# Капитал
equity_preprocess = historic_state.equity

liab_equity_preprocess = liab_preprocess + equity_preprocess
diff_preprocess = assets_preprocess - liab_equity_preprocess

print(f"\n📋 Баланс из HistoricState (препроцессинг):")
print(f"   Total Assets: ${assets_preprocess:,.0f}")
print(f"   Total Liabilities: ${liab_preprocess:,.0f}")
print(f"   Total Equity: ${equity_preprocess:,.0f}")
print(f"   Liabilities + Equity: ${liab_equity_preprocess:,.0f}")
print(f"   Diff (Assets - Liab+Equity): ${diff_preprocess:,.0f}")

if abs(diff_preprocess) < 1e-6:
    print(f"\n✅ Баланс сходится!")
else:
    print(f"\n❌ Баланс НЕ сходится! Разница: ${diff_preprocess:,.0f}")

# Сравнение с CANONICAL БД
if pd.notna(assets_canonical):
    print(f"\n📊 Сравнение с CANONICAL БД:")
    print(f"   Assets: ${assets_preprocess:,.0f} vs ${assets_canonical:,.0f} (diff: ${assets_preprocess - assets_canonical:,.0f})")
    print(f"   Liabilities: ${liab_preprocess:,.0f} vs ${liab_canonical:,.0f} (diff: ${liab_preprocess - liab_canonical:,.0f})")
    print(f"   Equity: ${equity_preprocess:,.0f} vs ${equity_canonical:,.0f} (diff: ${equity_preprocess - equity_canonical:,.0f})")


## 📊 ШАГ 5: Движок модели (ThreeStatementModel)


In [ ]:
# Загружаем модельhist_state, history, drivers, canonical = load_inputs(root=ROOT, company=COMPANY)model = ThreeStatementModel(**inputs)model.build()print("="*80)print("📊 ШАГ 5: ДВИЖОК МОДЕЛИ (ThreeStatementModel)")print("="*80)# Получаем баланс для 2024 года (последний исторический год)bs_2024 = model.BS[2024]# Суммируем активыassets_model = (    bs_2024.cash +    bs_2024.ar +    bs_2024.inventory +    bs_2024.other_current_assets +    bs_2024.ppe +    bs_2024.goodwill +    bs_2024.intangibles +    bs_2024.rou_asset +    bs_2024.other_non_current_assets)# Суммируем обязательстваliab_model = (    bs_2024.st_debt +    bs_2024.ap +    bs_2024.other_current_liabilities +    bs_2024.debt +    bs_2024.lease_liability +    bs_2024.other_non_current_liabilities)# Капиталequity_model = bs_2024.equityliab_equity_model = liab_model + equity_modeldiff_model = assets_model - liab_equity_modelprint(f"\n📋 Баланс из модели (2024 год):")print(f"   Total Assets: ${assets_model:,.0f}")print(f"   Total Liabilities: ${liab_model:,.0f}")print(f"   Total Equity: ${equity_model:,.0f}")print(f"   Liabilities + Equity: ${liab_equity_model:,.0f}")print(f"   Diff (Assets - Liab+Equity): ${diff_model:,.0f}")if abs(diff_model) < 1e-6:    print(f"\n✅ Баланс сходится!")else:    print(f"\n❌ Баланс НЕ сходится! Разница: ${diff_model:,.0f}")# Сравнение с препроцессингомprint(f"\n📊 Сравнение с препроцессингом:")print(f"   Assets: ${assets_model:,.0f} vs ${assets_preprocess:,.0f} (diff: ${assets_model - assets_preprocess:,.0f})")print(f"   Liabilities: ${liab_model:,.0f} vs ${liab_preprocess:,.0f} (diff: ${liab_model - liab_preprocess:,.0f})")print(f"   Equity: ${equity_model:,.0f} vs ${equity_preprocess:,.0f} (diff: ${equity_model - equity_preprocess:,.0f})")

In [ ]:
from IPython.display import display, Markdown

print("="*80)
print("📊 ИТОГОВАЯ СВОДКА ПРОВЕРКИ БАЛАНСА")
print("="*80)

summary_data = [
    {
        'Этап': 'Официальная отчетность',
        'Assets': f"${assets_official:,.0f}",
        'Liabilities': f"${liab_official:,.0f}",
        'Equity': f"${equity_official:,.0f}",
        'Liab+Equity': f"${liab_equity_official:,.0f}",
        'Diff': f"${diff_official:,.0f}",
        'Статус': '✅' if abs(diff_official) < 1e-6 else '❌'
    },
    {
        'Этап': 'RAW БД',
        'Assets': f"${assets_raw:,.0f}",
        'Liabilities': f"${liab_raw:,.0f}",
        'Equity': f"${equity_raw:,.0f}",
        'Liab+Equity': f"${liab_equity_raw:,.0f}",
        'Diff': f"${diff_raw:,.0f}",
        'Статус': '✅' if abs(diff_raw) < 1e-6 else '❌'
    },
]

# Добавляем CANONICAL БД если данные доступны
if pd.notna(assets_canonical) and pd.notna(liab_canonical) and pd.notna(equity_canonical):
    summary_data.append({
        'Этап': 'CANONICAL БД',
        'Assets': f"${assets_canonical:,.0f}",
        'Liabilities': f"${liab_canonical:,.0f}",
        'Equity': f"${equity_canonical:,.0f}",
        'Liab+Equity': f"${liab_equity_canonical:,.0f}",
        'Diff': f"${diff_canonical:,.0f}",
        'Статус': '✅' if abs(diff_canonical) < 1e-6 else '❌'
    })

summary_data.extend([
    {
        'Этап': 'Препроцессинг',
        'Assets': f"${assets_preprocess:,.0f}",
        'Liabilities': f"${liab_preprocess:,.0f}",
        'Equity': f"${equity_preprocess:,.0f}",
        'Liab+Equity': f"${liab_equity_preprocess:,.0f}",
        'Diff': f"${diff_preprocess:,.0f}",
        'Статус': '✅' if abs(diff_preprocess) < 1e-6 else '❌'
    },
    {
        'Этап': 'Движок модели',
        'Assets': f"${assets_model:,.0f}",
        'Liabilities': f"${liab_model:,.0f}",
        'Equity': f"${equity_model:,.0f}",
        'Liab+Equity': f"${liab_equity_model:,.0f}",
        'Diff': f"${diff_model:,.0f}",
        'Статус': '✅' if abs(diff_model) < 1e-6 else '❌'
    },
])

summary_df = pd.DataFrame(summary_data)
display(Markdown("#### 📊 Сводная таблица проверки баланса на всех этапах:"))
display(summary_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
]).set_properties(**{'font-size': '9px'}))

print("\n" + "="*80)
print("✅ ПРОВЕРКА ЗАВЕРШЕНА")
print("="*80)
print("\n💡 Интерпретация результатов:")
print("   ✅ - Баланс сходится (Assets = Liabilities + Equity)")
print("   ❌ - Баланс НЕ сходится (есть расхождение)")
print("\n   Если баланс не сходится на каком-то этапе, проверьте:")
print("   - Правильность маппинга метрик")
print("   - Правильность суммирования статей (combine_from)")
print("   - Потерю данных при преобразовании")
print("   - Логику препроцессинга или движка")

mart.close()
